# DBA-Fusion (RA-L 2024) -- learned/hybrid VIO branch of vio-bench

Runs DBA-Fusion in **monocular VIO mode (no GNSS / odometry)** on a list of sequences (`SEQUENCES` below: ROVER T265 and/or EuRoC),
converts each result to TUM (`traj_tum.txt`, IMU/body pose, seconds), evaluates it with the repo's `eval/evaluate.py`
(same metrics as the local OpenVINS / ORB-SLAM3 tables; ROVER: translation only + prism lever arm) and **uploads everything as ClearML artifacts**.

* **No Google Drive for data or results** (Drive is only an optional cache for the built gtsam/extension wheels, `USE_DRIVE_CACHE`).
  Code: `git clone` of the private GitHub repo with a token from Colab Secrets (fallback: upload the repo folder to `/content/vio-bench`).
  Data: downloaded straight to Colab's local disk `/content/data` (ROVER via HTTP Range from the HuggingFace mirror, EuRoC MH_01_easy from the HF zip).
  Results: ClearML project `vio-bench`, task `dbafusion/<dataset>/<seq>`, artifacts `traj_tum`, `log`, `run_stats`, `metrics`, `plots`.
* **ROVER** (T265 fisheye): DBA-Fusion needs pinhole images, so cam0 is rectified to 640x480 pinhole on the local disk (`tools/rover/undistort_asl.py`).
* Run top to bottom (Runtime -> GPU: T4 / L4 / A100). Every cell is idempotent: after a disconnect re-run from the top; data present in `/content/data`
  and sequences already on ClearML are skipped.
* Colab Secrets (Notebook access ON): `GITHUB_TOKEN` (fine-grained PAT, read-only Contents on the repo), `CLEARML_API_ACCESS_KEY`, `CLEARML_API_SECRET_KEY`,
  `CLEARML_API_HOST`, `CLEARML_WEB_HOST`, `CLEARML_FILES_HOST`. See `docs/ACCESS_SETUP.md`.
* Afterwards on the laptop: `pixi run fetch-colab` (pulls the artifacts into `results/dbafusion/...`, re-runs evaluation so DBA-Fusion joins the shared tables).

## 0. Parameters

In [ ]:
# ------------------------------- EDIT ME -------------------------------
# Sequences to run in this session, in order: (dataset, seq). Datasets: "rover" (T265, rectified in Colab) or "euroc" (sanity).
SEQUENCES = [
    ("rover", "garden_small_2024-05-29_1"),
    ("rover", "campus_small_2024-02-19"),
    ("rover", "park_2023-07-31"),
    # ("euroc", "MH_01_easy"),          # sanity check (~1.6 GB zip is pulled from HuggingFace by range request)
]
CONFIG_BY_DATASET = {"euroc": "euroc.yaml", "rover": "rover_t265.yaml"}   # methods/dbafusion/configs/<file>
OVERRIDES = {}                    # config overrides, e.g. {"dbaf.max_factors": 36, "stride": 3}
MAX_FRAMES = None                 # e.g. 100 for a quick smoke test (tagged "smoke" in ClearML, never picked up by `pixi run fetch-colab`)
RPE_DELTA_M = 10.0                # RPE delta in meters (same default as eval/evaluate.py)
FORCE_RERUN = False               # True: ignore existing results (local /content/results and finished ClearML tasks)
CLEARML_ENABLED = True
ROVER_RECT = {"width": 640, "height": 480, "hfov_deg": 100.0}   # must match configs/rover_t265.yaml (asserted below)
ROVER_MAX_SECONDS = None          # e.g. 20 to fetch only a 20 s prefix of every ROVER sequence (test); uses /content/data_smoke and /content/results_smoke

# Code: private GitHub repo, token = Colab Secret GITHUB_TOKEN (fine-grained PAT, Contents: read-only). Falls back to UPLOADED_REPO if that folder exists.
REPO_URL = "https://github.com/yahigz/robotics-vio-test.git"
REPO_REF = "main"                 # branch, tag or commit sha to run
UPLOADED_REPO = "/content/vio-bench"   # alternative: upload the whole repo folder here by hand (then no token is needed)

# Local (ephemeral) disk layout. DATA_ROOT follows CONTRACT.md: <DATA_ROOT>/<dataset>/<seq>/{asl,gt_tum.txt}
DATA_ROOT = "/content/data"
RESULTS_ROOT = "/content/results"
if ROVER_MAX_SECONDS: DATA_ROOT += "_smoke"; RESULTS_ROOT += "_smoke"    # prefix tests never mix with the real data/results
# Optional: Google Drive ONLY as a build cache (gtsam tarball + lietorch/droid_backends wheels + weights, a few hundred MB).
# False (or mount fails) -> everything is built from scratch in every new session (gtsam 20-45 min, extensions 8-20 min).
USE_DRIVE_CACHE = True
DRIVE_CACHE_DIR = "/content/drive/MyDrive/vio-bench-cache/dbafusion"

# Pins (do not change unless you know why)
DBAF_REPO = "https://github.com/GREAT-WHU/DBA-Fusion.git"
DBAF_COMMIT = "2c658a3be747247ed0b17af4aef42831ed5e7310"     # 2025-03-11 (main HEAD when pinned); lietorch submodule 0fa9ce8
GTSAM_REPO = "https://github.com/yuxuanzhou97/gtsam.git"      # DBA-Fusion's custom GTSAM fork
GTSAM_COMMIT = "c93ea5ba01dec1bf909ba9392fbe84975cb75a82"     # 2024-02-15 'Update README' = the version released with DBA-Fusion
WEIGHTS_GDRIVE_ID = "1PpqVt1H4maBa_GbPJp4NwxRsd9jk-elh"       # droid.pth (DROID-SLAM TartanAir weights), from DBA-Fusion README
# -----------------------------------------------------------------------

## 1. GPU check

In [ ]:
import subprocess, sys, os, json, time, shutil, platform
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No CUDA GPU: Runtime -> Change runtime type -> T4/L4/A100"
GPU_NAME = torch.cuda.get_device_name(0)
CAP = "%d.%d" % torch.cuda.get_device_capability(0)
PYV = "%d.%d" % sys.version_info[:2]
GPU_MEM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU {GPU_NAME} sm_{CAP.replace('.','')}  {GPU_MEM_GIB:.1f} GiB | python {PYV} | torch {torch.__version__} (CUDA {torch.version.cuda})")
print(subprocess.run("nvcc --version | tail -2", shell=True, capture_output=True, text=True).stdout)
if GPU_MEM_GIB < 11 and "dbaf.max_factors" not in OVERRIDES:
    print("NOTE: <11 GiB GPU; DBA-Fusion default (max_factors=48) needs ~10 GB. Consider OVERRIDES={'dbaf.max_factors': 36}.")

## 2. Code from GitHub (private repo, token from Colab Secrets) + optional Drive build cache

In [ ]:
import re
def _redact(s, secret):
    return s.replace(secret, "***") if secret else s

def git(*args, cwd=None, token=None, check=True):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f"git {args[0]} failed: " + _redact(r.stderr[-600:], token))
    return r.stdout.strip()

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

REPO = None
if os.path.isfile(f"{UPLOADED_REPO}/methods/dbafusion/scripts/run_dbafusion.py"):
    REPO = UPLOADED_REPO
    print("using the uploaded repo folder", UPLOADED_REPO)
else:
    token = get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("GITHUB_TOKEN is not set: add it in Colab Secrets (key icon, Notebook access ON; docs/ACCESS_SETUP.md), "
                           f"or upload the repo folder to {UPLOADED_REPO}")
    REPO = "/content/vio-bench-git"
    clean_url = REPO_URL
    auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    if not os.path.isdir(f"{REPO}/.git"):
        git("clone", "--no-checkout", auth_url, REPO, token=token)
        git("remote", "set-url", "origin", clean_url, cwd=REPO)      # do not leave the token in .git/config
    git("fetch", "-q", auth_url, "+refs/heads/*:refs/remotes/origin/*", "+refs/tags/*:refs/tags/*", cwd=REPO, token=token)
    ref = git("rev-parse", "--verify", "-q", f"origin/{REPO_REF}^{{commit}}", cwd=REPO, check=False) or \
          git("rev-parse", "--verify", "-q", f"{REPO_REF}^{{commit}}", cwd=REPO, check=False)
    assert ref, f"ref {REPO_REF!r} not found in {clean_url}"
    git("checkout", "-q", "--force", "--detach", ref, cwd=REPO)
    print("repo", clean_url, "@", ref[:10], f"({REPO_REF})")
METHOD_DIR, TOOLS = f"{REPO}/methods/dbafusion", f"{REPO}/tools"
SCRIPTS = f"{METHOD_DIR}/scripts"
EVAL_PY = f"{REPO}/eval/evaluate.py"
LEVER_CALIB = f"{TOOLS}/rover/calib/calib_t265.yaml"
for ds, _ in SEQUENCES:
    c = f"{METHOD_DIR}/configs/{CONFIG_BY_DATASET[ds]}"
    assert os.path.isfile(c), c
    if ds == "rover":
        assert os.path.isfile(f"{TOOLS}/rover/undistort_asl.py") and os.path.isfile(LEVER_CALIB), "tools/rover missing in the repo copy"
    if ds == "euroc":
        assert os.path.isfile(f"{TOOLS}/fetch_euroc_hf.py"), "tools/fetch_euroc_hf.py missing in the repo copy"

# tools/rover/*.py hard-code <repo>/data/rover as the data root -> point <repo>/data at the local data disk
os.makedirs(DATA_ROOT, exist_ok=True)
link = f"{REPO}/data"
if os.path.islink(link): os.unlink(link)
elif os.path.exists(link): os.rename(link, link + ".uploaded")
os.symlink(DATA_ROOT, link)

# Optional Drive build cache (NOT used for data or results)
CACHE = None
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE = DRIVE_CACHE_DIR; os.makedirs(CACHE, exist_ok=True)
    except Exception as e:
        print("!! Drive not mounted -> building without a persistent cache:", repr(e)[:200])
if CACHE is None:
    CACHE = "/content/cache/dbafusion"; os.makedirs(CACHE, exist_ok=True)
print("repo:", REPO, "| eval:", os.path.isfile(EVAL_PY), "| data:", DATA_ROOT, "| build cache:", CACHE, "| sequences:", SEQUENCES)

## 3. System + Python dependencies (torch is NOT touched: we build against Colab's torch)

In [ ]:
%%bash
set -e
apt-get -qq update >/dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y build-essential cmake ninja-build libboost-all-dev libeigen3-dev unzip wget curl rsync >/dev/null
# data tools need only numpy/opencv/pyyaml (preinstalled on Colab) + curl; evaluation needs evo; results go to ClearML
pip -q install evo clearml gdown pyyaml pyparsing setuptools wheel "pybind11>=2.13,<3" scipy
cmake --version | head -1

## 4. DBA-Fusion checkout (pinned) 
No numpy-quaternion / torch_scatter / open3d / h5py needed: our driver does not use them; `torch_scatter` is replaced by a pure-torch shim (`scripts/shims`).

In [ ]:
DBAF = "/content/DBA-Fusion"
def sh(cmd, env=None, every=25):
    """Run a shell command, stream a thinned-out copy of its output (Colab does not show child fds), raise on failure."""
    print("$", cmd)
    p = subprocess.Popen(["/bin/bash", "-c", "set -o pipefail; " + cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    tail = []
    for i, line in enumerate(p.stdout):
        tail = (tail + [line])[-60:]
        if i % every == 0 or "rror" in line: print(line, end="", flush=True)
    p.wait()
    if p.returncode:
        print("".join(tail)); raise RuntimeError(f"failed (exit {p.returncode}): {cmd}")
if not os.path.isdir(f"{DBAF}/.git"):
    sh(f"git clone {DBAF_REPO} {DBAF}")
sh(f"cd {DBAF} && git fetch -q origin && git checkout -q {DBAF_COMMIT} && git submodule update --init --recursive")
print(subprocess.run(f"cd {DBAF} && git rev-parse HEAD && git submodule status", shell=True, capture_output=True, text=True).stdout)

## 5. Custom GTSAM (python wrapper, fork required)
DBA-Fusion needs `BA2GTSAM`, `VelFactor`, `marginalizeOut`, `CombinedImuFactor.evaluateErrorCustom` ... that only exist in the author's GTSAM fork.
A fresh session builds it from source (~20-45 min on Colab's 2 vCPUs). With the optional Drive cache (`USE_DRIVE_CACHE`) the result is stored as a tarball and restored in seconds later.

In [ ]:
GTSAM_TAR = f"{CACHE}/gtsam-{GTSAM_COMMIT[:8]}-py{PYV}.tar.gz"

def gtsam_ok():
    r = subprocess.run([sys.executable, "-c",
        "import gtsam; assert all(hasattr(gtsam, n) for n in ('BA2GTSAM','VelFactor','marginalizeOut')); print('gtsam custom OK', gtsam.__file__)"],
        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:]); return r.returncode == 0

def build_gtsam(use_system_pybind):
    src = "/content/gtsam"
    if not os.path.isdir(f"{src}/.git"): sh(f"git clone {GTSAM_REPO} {src}")
    sh(f"cd {src} && git checkout -q {GTSAM_COMMIT}")
    sh(f"rm -rf {src}/build && mkdir {src}/build")
    pb = subprocess.run([sys.executable, "-m", "pybind11", "--cmakedir"], capture_output=True, text=True).stdout.strip()
    flags = (f"-DCMAKE_BUILD_TYPE=Release -DGTSAM_BUILD_PYTHON=ON -DGTSAM_PYTHON_VERSION={PYV} -DPYTHON_EXECUTABLE={sys.executable} "
             "-DGTSAM_BUILD_TESTS=OFF -DGTSAM_BUILD_EXAMPLES_ALWAYS=OFF -DGTSAM_BUILD_TIMING_ALWAYS=OFF "
             "-DGTSAM_BUILD_UNSTABLE=OFF -DGTSAM_UNSTABLE_BUILD_PYTHON=OFF -DGTSAM_WITH_TBB=OFF "
             "-DGTSAM_BUILD_WITH_MARCH_NATIVE=OFF -DCMAKE_POLICY_VERSION_MINIMUM=3.5 -DCMAKE_INSTALL_PREFIX=/usr/local")
    if use_system_pybind: flags += f" -Dpybind11_DIR={pb}"
    sh(f"cd {src}/build && cmake .. {flags}", every=5)
    sh(f"cd {src}/build && make -j$(nproc) && make install && ldconfig && make python-install", every=40)

if not gtsam_ok():
    if os.path.exists(GTSAM_TAR):
        sh(f"python {SCRIPTS}/gtsam_cache.py unpack {GTSAM_TAR}")
    if not gtsam_ok():
        t0 = time.time()
        try:
            build_gtsam(use_system_pybind=True)
        except RuntimeError as e:
            print("build with system pybind11 failed -> retrying with GTSAM's bundled pybind11:", e)
            build_gtsam(use_system_pybind=False)
        print(f"gtsam build took {(time.time()-t0)/60:.1f} min")
        assert gtsam_ok(), "gtsam built but custom API missing"
        sh(f"python {SCRIPTS}/gtsam_cache.py pack {GTSAM_TAR}")

## 6. Build `lietorch` + `droid_backends` against Colab's torch (wheels cached in the optional Drive cache)
Upstream `setup.py` only emits SASS up to sm_86 (no PTX), so it would not run on L4 (sm_89) etc.; `patch_build.py` regenerates the setup files for *this* GPU's
compute capability and replaces deprecated `Tensor.type()` in `AT_DISPATCH_*`. Wheels are keyed by python/torch/CUDA/arch.

In [ ]:
KEY = f"py{PYV}_torch{torch.__version__.replace('+','_')}_sm{CAP.replace('.','')}"
WHEELS = f"{CACHE}/wheels/{KEY}"; os.makedirs(WHEELS, exist_ok=True)

def ext_ok():
    code_ = ("import torch, lietorch, droid_backends; from lietorch import SE3; "
             "x=SE3.Random(8,device='cuda'); e=(x*x.inv()).log().abs().max().item(); assert e<1e-4, e; print('lietorch+droid_backends OK', e)")
    r = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-600:]); return r.returncode == 0

def build_ext(eigen_dir=None):
    env = dict(os.environ, MAX_JOBS="2", TORCH_CUDA_ARCH_LIST=CAP, CUDA_HOME=os.environ.get("CUDA_HOME", "/usr/local/cuda"))
    ed = f"--eigen_dir {eigen_dir}" if eigen_dir else ""
    sh(f"python {SCRIPTS}/patch_build.py --root {DBAF} --arch {CAP} {ed}")
    for name in ("setup_lietorch.py", "setup_droid_backends.py"):
        sh(f"cd {DBAF} && python {name} bdist_wheel -d {WHEELS}", env=env, every=15)

have = lambda p: any(f.startswith(p) for f in os.listdir(WHEELS))
if not ext_ok():
    if not (have("lietorch") and have("droid_backends")):
        t0 = time.time()
        try:
            build_ext()
        except Exception as e:
            print("default Eigen build failed, retrying with system Eigen 3.4:", e)
            build_ext("/usr/include/eigen3")
        print(f"extension build took {(time.time()-t0)/60:.1f} min")
    sh(f"pip -q install --force-reinstall --no-deps {WHEELS}/*.whl")
    assert ext_ok()
print("wheel cache:", os.listdir(WHEELS))

## 7. Weights + import smoke test

In [ ]:
WEIGHTS = f"{CACHE}/droid.pth"
if not os.path.exists(WEIGHTS):
    sh(f"gdown --id {WEIGHTS_GDRIVE_ID} -O {WEIGHTS}.part && mv {WEIGHTS}.part {WEIGHTS}")
print("weights:", WEIGHTS, f"{os.path.getsize(WEIGHTS)/2**20:.0f} MiB")
smoke = f"""
import sys; sys.path.insert(0, '{DBAF}/dbaf'); sys.path.append('{SCRIPTS}/shims')
import torch, matplotlib; matplotlib.use('Agg')
import gtsam, lietorch, droid_backends
from dbaf import DBAFusion
sd = torch.load('{WEIGHTS}', map_location='cpu'); assert any('update.weight.2.weight' in k for k in sd), 'unexpected checkpoint'
print('imports OK, checkpoint keys:', len(sd))
"""
r = subprocess.run([sys.executable, "-c", smoke], capture_output=True, text=True); print(r.stdout, r.stderr[-1500:])
assert r.returncode == 0

## 8. Per-sequence pipeline (functions)
For every `(dataset, seq)` in `SEQUENCES`: download data to local disk (skipped if present) -> (ROVER) rectify fisheye to pinhole -> run DBA-Fusion -> TUM ->
upload trajectory + log to ClearML -> evaluate with `eval/evaluate.py` (ROVER: `--trans-only --interp-est` + prism lever arm from `tools/rover/calib/calib_t265.yaml`) -> upload metrics/plots.
Data layout on the VM (CONTRACT.md): `/content/data/<dataset>/<seq>/{asl/mav0/{cam0,imu0},gt_tum.txt}`; ROVER also `/content/data/rover/calib_t265.yaml`.

In [ ]:
import yaml, glob
sys.path.insert(0, SCRIPTS)
import asl_to_dbafusion as A

def have_data(ds, seq):
    d = f"{DATA_ROOT}/{ds}/{seq}"
    return all(os.path.isfile(f"{d}/{p}") for p in ("asl/mav0/cam0/data.csv", "asl/mav0/imu0/data.csv", "gt_tum.txt"))

def ensure_data(ds, seq):
    """Idempotent download to local disk. ROVER: HF mirror by HTTP Range (T265 cam_left + imu + GT; tools/rover/fetch_subset.py) -> convert_asl.py
    (ASL layout, raw pieces deleted). EuRoC: only <seq>.zip is pulled out of the HF group zip (tools/fetch_euroc_hf.py)."""
    if have_data(ds, seq):
        print(f"data present: {DATA_ROOT}/{ds}/{seq}"); return
    if ds == "rover":
        # mono (cam_left) only; the 5 GB per-sequence limit of the laptop is irrelevant on Colab's ~100 GB disk
        ms = f"--max-seconds {ROVER_MAX_SECONDS}" if ROVER_MAX_SECONDS else ""
        sh(f"python {TOOLS}/rover/fetch_subset.py fetch {seq} --stereo no --max-gb 100 {ms}", every=5)
        sh(f"python {TOOLS}/rover/convert_asl.py {seq}", every=1000)       # moves PNGs into asl/, writes gt_tum.txt, deletes raw/
        shutil.copyfile(f"{TOOLS}/rover/calib/calib_t265.yaml", f"{DATA_ROOT}/rover/calib_t265.yaml")
    elif ds == "euroc":
        sh(f"python {TOOLS}/fetch_euroc_hf.py --seq {seq} --data-root {DATA_ROOT}", every=5)
    else:
        raise ValueError(ds)
    assert have_data(ds, seq), f"data incomplete after download: {DATA_ROOT}/{ds}/{seq}"
    sh(f"du -sh {DATA_ROOT}/{ds}/{seq}")

def stage_data(ds, seq):
    """Download if needed; ROVER: rectify the fisheye to pinhole on local disk. Returns (asl_dir_for_run, gt_path)."""
    ensure_data(ds, seq)
    asl_run = f"{DATA_ROOT}/{ds}/{seq}/asl"
    if ds == "rover":
        asl_run = f"/content/work/undist/{seq}"
        if not os.path.exists(f"{asl_run}/mav0/cam0/data.csv"):     # data.csv is written last -> completion marker
            r = ROVER_RECT
            sh(f"python {TOOLS}/rover/undistort_asl.py {DATA_ROOT}/rover/{seq} --size {r['width']} {r['height']} --hfov {r['hfov_deg']} "
               f"--out {asl_run} --force", every=1000)
        n = len(os.listdir(f"{asl_run}/mav0/cam0/data")); print(f"rectified frames: {n}")
    return asl_run, f"{DATA_ROOT}/{ds}/{seq}/gt_tum.txt"

## 9. ClearML (results are uploaded as artifacts; no Drive)

In [ ]:
HAVE_CLEARML = False
if CLEARML_ENABLED:
    try:
        for k in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY", "CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
            v = get_secret(k)
            assert v, f"secret {k} missing"
            os.environ[k] = v
        import clearml
        HAVE_CLEARML = True
    except Exception as e:
        print("!! ClearML disabled (results stay on the VM only!):", repr(e))

def clearml_done(ds, seq):
    """True if a (non-smoke) ClearML task of this sequence already has a traj_tum artifact."""
    if not HAVE_CLEARML: return False
    try:
        from clearml import Task
        for t in Task.get_tasks(project_name="vio-bench", task_name=f"^dbafusion/{ds}/{seq}$"):
            if "traj_tum" in t.artifacts and "smoke" not in (t.get_tags() or []): return True
    except Exception as e:
        print("!! could not query ClearML:", repr(e)[:200])
    return False

def start_task(ds, seq, cfg):
    if not HAVE_CLEARML: return None
    try:
        from clearml import Task
        tags = ["dbafusion", ds, seq] + (["smoke"] if MAX_FRAMES is not None or ROVER_MAX_SECONDS else [])
        task = Task.init(project_name="vio-bench", task_name=f"dbafusion/{ds}/{seq}", reuse_last_task_id=False,
                         tags=tags, auto_connect_frameworks={"matplotlib": False, "pytorch": False})
        task.connect(cfg, name="config")
        task.connect({"dataset": ds, "seq": seq, "overrides": OVERRIDES, "max_frames": MAX_FRAMES, "rover_max_seconds": ROVER_MAX_SECONDS,
                      "rpe_delta_m": RPE_DELTA_M, "repo_ref": REPO_REF, "dbaf_commit": DBAF_COMMIT, "gtsam_commit": GTSAM_COMMIT,
                      "gpu": GPU_NAME, "compute_capability": CAP, "torch": torch.__version__, "cuda": torch.version.cuda,
                      "python": platform.python_version()}, name="run")
        print("ClearML task:", task.get_output_log_web_page())
        return task
    except Exception as e:
        print("!! ClearML task failed:", repr(e)); return None

def upload(task, name, path):
    """Upload a file or folder as artifact `name` (waits for the upload)."""
    if task is None or not os.path.exists(path): return
    try:
        task.upload_artifact(name, artifact_object=path, wait_on_upload=True)
    except Exception as e:
        print(f"!! artifact {name} upload failed:", repr(e)[:300])

## 10. Run DBA-Fusion (VIO, no GNSS) for one sequence

In [ ]:
def run_dbafusion(ds, seq, asl_run, cfg_path, cfg):
    """Run the driver; results go to the local /content/results (uploaded to ClearML by the caller). Returns (traj_path, stats, res_dir, smoke)."""
    smoke = MAX_FRAMES is not None or bool(ROVER_MAX_SECONDS)
    RES = f"{RESULTS_ROOT}/dbafusion{'_smoke' if smoke else ''}/{ds}/{seq}"; os.makedirs(RES, exist_ok=True)
    WORK = f"/content/work/{ds}/{seq}"; os.makedirs(WORK, exist_ok=True)
    TRAJ = f"{RES}/traj_tum.txt"
    RAW, STATS, LOG = f"{WORK}/raw_result.txt", f"{WORK}/run_stats.json", f"{WORK}/log.txt"
    if os.path.exists(TRAJ) and not FORCE_RERUN and not smoke:
        print("traj_tum.txt exists locally -> skipping run (FORCE_RERUN=True to redo)")
        return TRAJ, (json.load(open(f"{RES}/run_stats.json")) if os.path.exists(f"{RES}/run_stats.json") else {}), RES, smoke
    cmd = [sys.executable, "-u", f"{SCRIPTS}/run_dbafusion.py", "--dbaf_root", DBAF, "--asl_dir", asl_run, "--config", cfg_path,
           "--weights", WEIGHTS, "--out", RAW, "--stats", STATS]
    for k, v in OVERRIDES.items(): cmd += ["--set", f"{k}={json.dumps(v)}"]
    if MAX_FRAMES: cmd += ["--max_frames", str(MAX_FRAMES)]
    t0 = time.time()
    with open(LOG, "w") as lf:
        p = subprocess.Popen(cmd, cwd=DBAF, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            lf.write(line); lf.flush()
            if any(s in line for s in ("[adapter]", "[run]", "[stats]", "[done]", "Error", "error", "Traceback", "IMU excitation")): print(line, end="")
        p.wait()
    shutil.copy(LOG, f"{RES}/log.txt")
    if p.returncode:
        print(open(LOG).read()[-4000:]); raise RuntimeError(f"run_dbafusion failed (exit {p.returncode}); full log: {RES}/log.txt")
    stats = json.load(open(STATS)); stats["total_wall_time_incl_startup_s"] = time.time() - t0
    import to_tum
    traj_out = TRAJ
    info = to_tum.convert(RAW, traj_out); print("TUM:", info)
    stats.update(info)
    json.dump(stats, open(f"{RES}/run_stats.json", "w"), indent=2)
    shutil.copy(RAW, f"{RES}/raw_result.txt")
    shutil.copy(cfg_path, f"{RES}/config_used.yaml"); yaml.safe_dump(cfg, open(f"{RES}/config_effective.yaml", "w"))
    return traj_out, stats, RES, smoke

def upload_run(task, traj, res):
    """Trajectory + log + stats are uploaded right after the run (before evaluation), so a failing eval never loses the result."""
    upload(task, "traj_tum", traj)
    upload(task, "log", f"{res}/log.txt")
    upload(task, "run_stats", f"{res}/run_stats.json")

## 11. Evaluate and report to ClearML
Uses the repo's `eval/evaluate.py` (identical to the laptop evaluation): SE(3) alignment, RPE delta in meters. ROVER GT is prism position only (~2 Hz):
`--trans-only --interp-est --lever-arm-calib` (estimate interpolated at GT times, IMU pose moved to the prism). Other datasets: evaluate.py defaults.
(Fallback `scripts/eval_traj.py` if the repo copy has no `eval/`; that path has no lever arm.) Metrics are uploaded as artifact `metrics`, the evaluation folder (png/pdf) as `plots`.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

def evaluate_seq(ds, seq, traj, gt, stats, res, smoke, task):
    assert os.path.exists(gt), f"ground truth missing: {gt}"
    sfx = ""
    if os.path.isfile(EVAL_PY):
        out_root = f"{res}/eval{sfx}"
        cmd = [sys.executable, EVAL_PY, "--gt", gt, "--est", f"dbafusion={traj}", "--dataset", ds, "--seq", seq,
               "--out-root", out_root, "--delta", str(RPE_DELTA_M)]
        if ds == "rover":
            cmd += ["--trans-only", "--interp-est", "--lever-arm-calib", LEVER_CALIB]
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout[-2500:], r.stderr[-1500:])
        r.check_returncode()
        odir = f"{out_root}/{ds}/{seq}"
        m = json.load(open(f"{odir}/metrics.json"))["methods"]["dbafusion"]
        pngs = sorted(glob.glob(f"{odir}/*.png"))
        for pth in pngs[:3]:
            plt.figure(figsize=(7, 5)); plt.imshow(mpimg.imread(pth)); plt.axis("off"); plt.title(os.path.basename(pth)); plt.show()
        if task is not None:
            lg = task.get_logger()
            for k, v in m["ape_trans_m"].items(): lg.report_scalar("APE_translation_m", k, float(v), iteration=0)
            for k, v in (m["rpe"].get("trans_m") or {}).items(): lg.report_scalar("RPE_translation_m", k, float(v), iteration=0)
            lg.report_scalar("trajectory", "n_matched", float(m["n_matched"]), iteration=0)
            lg.report_scalar("trajectory", "gt_path_length_m", float(m["gt_path_length_m"]), iteration=0)
            for pth in pngs: lg.report_image("plots", os.path.basename(pth)[:-4], iteration=0, local_path=pth)
            upload(task, "metrics", f"{odir}/metrics.json")
            upload(task, "plots", odir)
        summary = {"ape_rmse_m": m["ape_trans_m"]["rmse"], "n_matched": m["n_matched"], "gt_path_length_m": m["gt_path_length_m"]}
    else:
        print("!! eval/evaluate.py not in the repo copy -> legacy eval_traj (no lever arm; ROVER numbers NOT comparable)")
        import eval_traj
        metrics, figs = eval_traj.evaluate(gt, traj, rpe_delta_m=RPE_DELTA_M)
        json.dump(metrics, open(f"{res}/eval_metrics{sfx}.json", "w"), indent=2)
        for n, f in figs.items(): f.savefig(f"{res}/{n}{sfx}.png", dpi=130)
        summary = {"ape_rmse_m": metrics["ape_trans_m"]["rmse"], "n_matched": metrics["n_assoc"], "gt_path_length_m": metrics["path_length_m"]}
        if task is not None:
            lg = task.get_logger()
            for k, v in metrics["ape_trans_m"].items(): lg.report_scalar("APE_translation_m", k, float(v), iteration=0)
    if task is not None:
        lg = task.get_logger()
        for s in ("wall_time_s", "fps", "gpu_max_allocated_gib", "gpu_max_reserved_gib", "frames_processed", "total_wall_time_incl_startup_s"):
            if stats.get(s) is not None: lg.report_scalar("runtime", s, float(stats[s]), iteration=0)
        task.flush(wait_for_uploads=True)
    return summary

## 12. Run all sequences
Each sequence is independent: a failure is recorded and the loop continues. A sequence is skipped if its result already exists (local `/content/results` or a finished ClearML task).
Results: ClearML project `vio-bench`, task `dbafusion/<dataset>/<seq>`. On the laptop: `pixi run fetch-colab`.

In [ ]:
SUMMARY = {}
SMOKE = MAX_FRAMES is not None or bool(ROVER_MAX_SECONDS)
for ds, seq in SEQUENCES:
    print("\n" + "=" * 100 + f"\n{ds}/{seq}\n" + "=" * 100)
    task = None
    try:
        if not FORCE_RERUN and not SMOKE and (os.path.exists(f"{RESULTS_ROOT}/dbafusion/{ds}/{seq}/traj_tum.txt") or clearml_done(ds, seq)):
            print("result already exists (local or on ClearML) -> skip (FORCE_RERUN=True to redo)")
            SUMMARY[f"{ds}/{seq}"] = {"skipped": "result exists"}; continue
        cfg_path = f"{METHOD_DIR}/configs/{CONFIG_BY_DATASET[ds]}"
        cfg = A.load_config(cfg_path, OVERRIDES)
        if ds == "rover":   # the config must describe exactly the rectified pinhole the notebook produces
            import math
            r = ROVER_RECT; fx = (r["width"] / 2) / math.tan(math.radians(r["hfov_deg"]) / 2)
            assert abs(cfg["camera"]["fx"] - fx) < 1e-3 and cfg["camera"]["width"] == r["width"], "ROVER_RECT does not match configs/rover_t265.yaml"
        asl_run, gt = stage_data(ds, seq)
        sh(f"python {SCRIPTS}/asl_to_dbafusion.py --asl_dir {asl_run} --config {cfg_path} --check")
        task = start_task(ds, seq, cfg)
        traj, stats, res, smoke = run_dbafusion(ds, seq, asl_run, cfg_path, cfg)
        upload_run(task, traj, res)
        SUMMARY[f"{ds}/{seq}"] = evaluate_seq(ds, seq, traj, gt, stats, res, smoke, task)
    except Exception as e:
        import traceback; traceback.print_exc()
        SUMMARY[f"{ds}/{seq}"] = {"error": repr(e)}
        upload(task, "log_failed", f"/content/work/{ds}/{seq}/log.txt")
    finally:
        if task is not None: task.close()
print(json.dumps(SUMMARY, indent=1))

In [ ]:
print("local results:", f"{RESULTS_ROOT}/dbafusion")
for ds, seq in SEQUENCES:
    d = f"{RESULTS_ROOT}/dbafusion/{ds}/{seq}"
    print(f"  {ds}/{seq}:", sorted(os.listdir(d)) if os.path.isdir(d) else "-")
if HAVE_CLEARML:
    print("Results are on ClearML (project vio-bench, tasks dbafusion/<dataset>/<seq>). Next (laptop): pixi run fetch-colab")
else:
    # no ClearML: the VM disk is the only copy -> hand the results to the browser
    shutil.make_archive("/content/dbafusion_results", "zip", f"{RESULTS_ROOT}/dbafusion")
    try:
        from google.colab import files; files.download("/content/dbafusion_results.zip")
    except Exception as e:
        print("download /content/dbafusion_results.zip from the Files pane:", repr(e)[:100])